In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPyImage, display

# 1. Load your trained model
model = YOLO("best.pt")

# 2. Run prediction on an image
results = model(
    source="local_path.png",
    imgsz=640,
    conf=0.25,
    save=True,
    device="cpu"
)


image 1/1 /home/pawan/Documents/NIDAR/VisDrone2019-MOT-test-dev/sequences/uav0000161_00000_v/0000001.jpg: 384x640 25 persons, 289.7ms
Speed: 4.3ms preprocess, 289.7ms inference, 16.7ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /home/pawan/Documents/NIDAR/runs/detect/predict3


In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont

ROOT_DIR = "VisDrone2019-MOT-train"
ROOT_ANNOTATIONS_DIR = os.path.join(ROOT_DIR, "annotations")

# New YOLO annotations you generated with the previous script
NEW_ANN_ROOT = os.path.join("train", "annotations")

# VisDrone categories: 1 = pedestrian, 2 = people
PERSON_CATEGORIES = {1, 2}


def load_original_boxes(seq_name, frame_id, img_size):
    """
    Load VisDrone MOT boxes for a specific sequence and frame.
    Returns list of (x1, y1, x2, y2).
    """
    anno_path = os.path.join(ROOT_ANNOTATIONS_DIR, seq_name + ".txt")
    boxes = []
    if not os.path.isfile(anno_path):
        print(f"[ERROR] Original annotation file not found: {anno_path}")
        return boxes

    with open(anno_path, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 8:
                continue

            f_id = int(parts[0])
            if f_id != frame_id:
                continue

            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])
            cat = int(parts[7])

            if cat not in PERSON_CATEGORIES:
                continue

            x1 = x
            y1 = y
            x2 = x + w
            y2 = y + h
            boxes.append((x1, y1, x2, y2))

    return boxes


def load_yolo_boxes(seq_name, frame_id, img_size):
    """
    Load YOLO boxes (your converted annotations) for a specific sequence and frame.
    Returns list of (x1, y1, x2, y2).
    """
    img_w, img_h = img_size
    label_path = os.path.join(NEW_ANN_ROOT, seq_name, f"{frame_id:07d}.txt")
    boxes = []

    if not os.path.isfile(label_path):
        print(f"[WARN] YOLO annotation file not found: {label_path}")
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            # class_id = int(parts[0])  # we don't need it for comparison here
            x_c = float(parts[1]) * img_w
            y_c = float(parts[2]) * img_h
            w = float(parts[3]) * img_w
            h = float(parts[4]) * img_h

            x1 = x_c - w / 2.0
            y1 = y_c - h / 2.0
            x2 = x_c + w / 2.0
            y2 = y_c + h / 2.0
            boxes.append((x1, y1, x2, y2))

    return boxes


def draw_boxes(image, boxes, color, label):
    """
    Draw rectangles on a copy of image and return it.
    """
    img = image.copy()
    draw = ImageDraw.Draw(img)
    for (x1, y1, x2, y2) in boxes:
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
    # Add label text at top-left
    try:
        font = ImageFont.load_default()
        draw.text((10, 10), label, fill=color, font=font)
    except:
        draw.text((10, 10), label, fill=color)
    return img


def iou(boxA, boxB):
    """
    Compute IoU between two boxes: (x1, y1, x2, y2).
    """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH

    if interArea == 0:
        return 0.0

    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return interArea / float(areaA + areaB - interArea + 1e-6)


def compare_boxes(orig_boxes, yolo_boxes, iou_threshold=0.999):
    """
    Compare two sets of boxes. They should be identical after conversion.
    Prints IoU for each match.
    """
    print(f"[INFO] #orig boxes = {len(orig_boxes)}, #yolo boxes = {len(yolo_boxes)}")

    if len(orig_boxes) != len(yolo_boxes):
        print("[WARN] Box counts do NOT match!")

    # Sort both lists by (x1, y1) to make pairing deterministic
    orig_sorted = sorted(orig_boxes, key=lambda b: (b[0], b[1]))
    yolo_sorted = sorted(yolo_boxes, key=lambda b: (b[0], b[1]))

    for i, (b0, b1) in enumerate(zip(orig_sorted, yolo_sorted)):
        iou_val = iou(b0, b1)
        print(f"  Box {i}: IoU = {iou_val:.6f}")
        if iou_val < iou_threshold:
            print("   -> MISMATCH (IoU < threshold)")
        else:
            print("   -> OK")


def debug_one_frame(seq_name, frame_id):
    """
    For a given sequence and frame:
      - load image
      - draw VisDrone boxes
      - draw YOLO boxes
      - save two debug images
      - print IoUs
    """
    img_path = os.path.join(ROOT_DIR, "sequences", seq_name, f"{frame_id:07d}.jpg")
    if not os.path.isfile(img_path):
        print(f"[ERROR] Image not found: {img_path}")
        return

    image = Image.open(img_path).convert("RGB")
    img_w, img_h = image.size

    orig_boxes = load_original_boxes(seq_name, frame_id, (img_w, img_h))
    yolo_boxes = load_yolo_boxes(seq_name, frame_id, (img_w, img_h))

    print(f"[INFO] Found {len(orig_boxes)} original boxes, {len(yolo_boxes)} YOLO boxes")

    img_orig = draw_boxes(image, orig_boxes, color="red", label="Original VisDrone")
    img_yolo = draw_boxes(image, yolo_boxes, color="lime", label="YOLO converted")

    # Save debug images
    out_orig = f"debug_original_{seq_name}_{frame_id:07d}.jpg"
    out_yolo = f"debug_yolo_{seq_name}_{frame_id:07d}.jpg"
    img_orig.save(out_orig)
    img_yolo.save(out_yolo)
    print(f"[INFO] Saved {out_orig} and {out_yolo}")

    # Compare numerically
    compare_boxes(orig_boxes, yolo_boxes)


if __name__ == "__main__":
    # Example: change these to a real sequence and frame that exists
    SEQ_NAME = "uav0000013_00000_v"
    FRAME_ID = 1   # first frame

    debug_one_frame(SEQ_NAME, FRAME_ID)


[INFO] Found 15 original boxes, 15 YOLO boxes
[INFO] Saved debug_original_uav0000013_00000_v_0000001.jpg and debug_yolo_uav0000013_00000_v_0000001.jpg
[INFO] #orig boxes = 15, #yolo boxes = 15
  Box 0: IoU = 0.999973
   -> OK
  Box 1: IoU = 0.999986
   -> OK
  Box 2: IoU = 0.999950
   -> OK
  Box 3: IoU = 0.999974
   -> OK
  Box 4: IoU = 0.999962
   -> OK
  Box 5: IoU = 0.999971
   -> OK
  Box 6: IoU = 0.999972
   -> OK
  Box 7: IoU = 0.999937
   -> OK
  Box 8: IoU = 0.999973
   -> OK
  Box 9: IoU = 0.999975
   -> OK
  Box 10: IoU = 0.999969
   -> OK
  Box 11: IoU = 0.999967
   -> OK
  Box 12: IoU = 0.999962
   -> OK
  Box 13: IoU = 0.999985
   -> OK
  Box 14: IoU = 0.999950
   -> OK
